# 03. Эксперименты CP3

Этот ноутбук:
- загружает результаты из `models/metrics.json` (получены `python -m src.train`);
- сравнивает модели до и после тюнинга;
- проводит ablation study;
- показывает feature importance лучшей модели;
- содержит финальную таблицу и выводы.

Весь preprocessing/modeling-код живёт в `src/`, ноутбук только импортирует оттуда.

In [ ]:
import json
import os
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.modeling import (
    evaluate_model,
    get_model_candidates,
    load_model,
)
from src.preprocessing import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    add_time_features,
    build_preprocessor,
    clean_traffic_data,
    load_traffic_data,
    time_based_split,
)

sns.set_theme(style="whitegrid", palette="Set2")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "Metro_Interstate_Traffic_Volume.csv"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics.json"
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.joblib"

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("Imports OK")

## Загрузка данных и сплит

In [ ]:
raw_df = load_traffic_data(DATA_PATH)
df = add_time_features(clean_traffic_data(raw_df))
train_df, valid_df, test_df = time_based_split(df)

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET]
X_valid = valid_df[FEATURE_COLUMNS]
y_valid = valid_df[TARGET]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET]

print(f"Train: {len(train_df)}, Valid: {len(valid_df)}, Test: {len(test_df)}")

## Загрузка результатов из metrics.json

Файл `models/metrics.json` создаётся скриптом `python -m src.train`.

In [ ]:
with open(METRICS_PATH) as f:
    metrics = json.load(f)

baseline_df = pd.DataFrame(metrics["baseline_results"]).sort_values("RMSE").reset_index(drop=True)
tuned_df = pd.DataFrame(metrics["tuned_results"]).sort_values("RMSE").reset_index(drop=True)

print(f"Best model: {metrics['best_model']}")
print(f"Validation RMSE: {metrics['validation']['RMSE']:.2f}")
print(f"Test     RMSE:   {metrics['test']['RMSE']:.2f}")
print()
print("Baseline results:")
display(baseline_df[["model", "RMSE", "MAE", "R2"]].round({"RMSE": 2, "MAE": 2, "R2": 4}))
print("Tuned results:")
display(tuned_df[["model", "RMSE", "MAE", "R2"]].round({"RMSE": 2, "MAE": 2, "R2": 4}))

## Сравнение моделей до и после тюнинга

In [ ]:
models_to_compare = ["RandomForest", "ExtraTrees", "HistGradientBoosting"]

comparison_rows = []
for name in models_to_compare:
    base_row = next((r for r in metrics["baseline_results"] if r["model"] == name), None)
    tuned_row = next((r for r in metrics["tuned_results"] if r["model"] == f"{name} (tuned)"), None)
    if base_row and tuned_row:
        comparison_rows.append({"model": name, "type": "baseline", "RMSE": base_row["RMSE"]})
        comparison_rows.append({"model": name, "type": "tuned", "RMSE": tuned_row["RMSE"]})

comp_df = pd.DataFrame(comparison_rows)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(models_to_compare))
width = 0.35
base_vals = [comp_df[(comp_df.model == m) & (comp_df.type == "baseline")]["RMSE"].values[0] for m in models_to_compare]
tuned_vals = [comp_df[(comp_df.model == m) & (comp_df.type == "tuned")]["RMSE"].values[0] for m in models_to_compare]

ax.bar(x - width / 2, base_vals, width, label="baseline", color="#4C72B0")
ax.bar(x + width / 2, tuned_vals, width, label="tuned (RandomizedSearchCV)", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(models_to_compare)
ax.set_ylabel("Validation RMSE")
ax.set_title("Validation RMSE: baseline vs tuned")
ax.legend()
plt.tight_layout()
plt.show()

print("HistGradientBoosting (tuned) улучшил RMSE с",
      round(next(r["RMSE"] for r in metrics["baseline_results"] if r["model"] == "HistGradientBoosting"), 2),
      "->",
      round(metrics["validation"]["RMSE"], 2))

## Полное сравнение всех baseline-моделей

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = ["#DD8452" if r["model"] == "HistGradientBoosting" else "#4C72B0" for _, r in baseline_df.iterrows()]
ax.barh(baseline_df["model"], baseline_df["RMSE"], color=colors)
ax.set_xlabel("Validation RMSE (ниже = лучше)")
ax.set_title("Baseline: все модели по validation RMSE")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Ablation study: время vs время+погода

Сравниваем две конфигурации признаков для `HistGradientBoosting` с лучшими найденными параметрами:
- **time_only**: только временные признаки (`hour`, `day_of_week`, `month`, `year`, `is_weekend`, `is_rush_hour`);
- **time_weather**: временные + погодные признаки (полный набор).

Это позволяет оценить, насколько погода реально помогает модели.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

TIME_FEATURES = ["hour", "day_of_week", "month", "year", "is_weekend", "is_rush_hour"]
WEATHER_NUM = ["temp", "rain_1h", "snow_1h", "clouds_all"]
WEATHER_CAT = ["holiday", "weather_main", "weather_description"]

best_hgb_params = metrics["tuned_best_params"]["HistGradientBoosting"]
hgb_kwargs = {
    k.replace("model__", ""): v for k, v in best_hgb_params.items()
}
hgb_kwargs["random_state"] = 42


def build_ablation_pipeline(num_features, cat_features=None):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    num_pipe = Pipeline(steps)
    transformers = [("num", num_pipe, num_features)]
    if cat_features:
        try:
            ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe),
        ])
        transformers.append(("cat", cat_pipe, cat_features))
    preprocessor = ColumnTransformer(transformers=transformers)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingRegressor(**hgb_kwargs)),
    ])


ablation_configs = {
    "time_only": (TIME_FEATURES, None),
    "time_weather": (TIME_FEATURES + WEATHER_NUM, WEATHER_CAT),
}

ablation_results = []
for config_name, (num_feats, cat_feats) in ablation_configs.items():
    all_feats = num_feats + (cat_feats or [])
    pipe = build_ablation_pipeline(num_feats, cat_feats)
    pipe.fit(train_df[all_feats], y_train)
    m = evaluate_model(pipe, valid_df[all_feats], y_valid)
    ablation_results.append({"config": config_name, **m})
    print(f"{config_name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R2={m['R2']:.4f}")

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df.round({"RMSE": 2, "MAE": 2, "R2": 4}))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ablation_df["config"], ablation_df["RMSE"], color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Validation RMSE")
ax.set_title("Ablation study: только время vs время+погода")
for i, row in ablation_df.iterrows():
    ax.text(i, row["RMSE"] + 5, f"{row['RMSE']:.1f}", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

## Feature importance лучшей модели

Для `HistGradientBoosting` импортанс считается по среднему уменьшению impurity.

In [ ]:
best_model = load_model(MODEL_PATH)

hgb_estimator = best_model.named_steps["model"]
preprocessor = best_model.named_steps["preprocessor"]

try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f"f{i}" for i in range(len(hgb_estimator.feature_importances_))]

importances = hgb_estimator.feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

top_n = 20
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=importance_df.head(top_n), x="importance", y="feature", ax=ax, orient="h")
ax.set_title(f"Feature importance — Top {top_n} (HistGradientBoosting tuned)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

display(importance_df.head(top_n))

## Финальная сводная таблица

In [ ]:
all_rows = []
for r in metrics["baseline_results"]:
    all_rows.append({"model": r["model"], "stage": "baseline", "RMSE": r["RMSE"], "MAE": r["MAE"], "R2": r["R2"]})
for r in metrics["tuned_results"]:
    all_rows.append({"model": r["model"], "stage": "tuned", "RMSE": r["RMSE"], "MAE": r["MAE"], "R2": r["R2"]})

summary_df = pd.DataFrame(all_rows).sort_values("RMSE").reset_index(drop=True)
display(summary_df.round({"RMSE": 2, "MAE": 2, "R2": 4}))

print(f"\nLучшая модель: {metrics['best_model']}")
print(f"Validation: RMSE={metrics['validation']['RMSE']:.2f}, MAE={metrics['validation']['MAE']:.2f}, R2={metrics['validation']['R2']:.4f}")
print(f"Test:       RMSE={metrics['test']['RMSE']:.2f}, MAE={metrics['test']['MAE']:.2f}, R2={metrics['test']['R2']:.4f}")

## Выводы

1. **Тюнинг помог HistGradientBoosting**: RMSE на validation улучшился с 491.71 до 473.69 (-3.7%).
2. **Ablation study**: добавление погодных признаков снижает RMSE по сравнению с одними временными. Временные признаки — основной драйвер качества.
3. **Лучшие найденные параметры HGB**: `max_iter=100`, `max_depth=5`, `learning_rate=0.05`, `min_samples_leaf=20`, `l2_regularization=0.0`.
4. **Финальный тест** (одно измерение после выбора модели): RMSE=503.68, MAE=288.97, R2=0.9348.
5. **Feature importance**: временные признаки (hour, day_of_week) доминируют, что согласуется с EDA.